In [2]:
import pandas as pd
import numpy as np
from shapely import wkt
import geopandas as gpd
import requests, time
from rapidfuzz import fuzz

In [3]:
ind_landmarks_df = pd.read_csv("source_csvs/Individual_Landmark_Sites_20260725.csv")
neighborhoods_df = pd.read_csv("source_csvs/2020_Neighborhood_Tabulation_Areas_(NTAs)_20260802.csv")

In [4]:
neighborhoods_df['the_geom'] = neighborhoods_df['the_geom'].apply(wkt.loads)
ind_landmarks_df['the_geom'] = ind_landmarks_df['the_geom'].apply(wkt.loads)

In [5]:
# neighborhoods_gdf = gpd.GeoDataFrame(neighborhoods_df, geometry='the_geom', crs='EPSG:2263')
landmarks_gdf = gpd.GeoDataFrame(ind_landmarks_df, geometry='the_geom', crs='EPSG:2263')

In [6]:
# This new dataset is lat/lon -> EPSG:4326
neighborhoods_gdf = gpd.GeoDataFrame(neighborhoods_df, geometry='the_geom', crs='EPSG:4326')

# Reproject neighborhoods into the landmarks' CRS (EPSG:2263) so they match
neighborhoods_gdf = neighborhoods_gdf.to_crs('EPSG:2263')

In [7]:
# landmarks_gdf = landmarks_gdf.to_crs(neighborhoods_gdf.crs)

# 4. For each landmark, find the neighborhood it falls inside
def find_neighborhood(landmark_geom):
    matches = neighborhoods_gdf[neighborhoods_gdf.contains(landmark_geom)]
    if not matches.empty:
        return matches.iloc[0]['NTAName']  # adjust to your actual name column
    return None


In [8]:
HEADERS = {"User-Agent": "landmark-fame-project (sean.cary62@gmail.com)"}  

def geosearch_wikipedia(lat, lon, radius_m=150, limit=5):
    r = requests.get("https://en.wikipedia.org/w/api.php", params={
        "action": "query", "list": "geosearch",
        "gscoord": f"{lat}|{lon}", "gsradius": radius_m,
        "gslimit": limit, "format": "json"
    }, headers=HEADERS)
    return r.json().get("query", {}).get("geosearch", [])

In [9]:
def best_match(landmark_name, candidates):
    if not candidates:
        return None, 0
    scored = [(c, fuzz.token_sort_ratio(landmark_name, c["title"])) for c in candidates]
    scored.sort(key=lambda x: -x[1])
    return scored[0]  # (candidate dict, similarity score 0-100)

In [10]:
def resolve_title(title):
    r = requests.get("https://en.wikipedia.org/w/api.php", params={
        "action": "query", "titles": title, "redirects": 1, "format": "json"
    }, headers=HEADERS)
    pages = r.json().get("query", {}).get("pages", {})
    return next(iter(pages.values())).get("title", title)

def get_pageviews(title, start="20240101", end="20241231"):
    print(title)
    title_enc = resolve_title(title).replace(" ", "_")
    r = requests.get(
        f"https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/"
        f"en.wikipedia.org/all-access/user/{title_enc}/monthly/{start}/{end}",
        headers=HEADERS
    )
    if r.status_code != 200:
        return None
    return sum(item["views"] for item in r.json().get("items", []))

def resolve_title_and_url(title):
    r = requests.get("https://en.wikipedia.org/w/api.php", params={
        "action": "query", "titles": title, "redirects": 1,
        "prop": "info", "inprop": "url", "format": "json"
    }, headers=HEADERS)
    pages = r.json().get("query", {}).get("pages", {})
    page = next(iter(pages.values()))
    return {
        "title": page.get("title", title),
        "url": page.get("fullurl")  # None if the page doesn't exist
    }

In [11]:
def match_and_score(row, radius_m=150, name_threshold=70):
    candidates = geosearch_wikipedia(row["Latitude"], row["Longitude"], radius_m=radius_m)
    match, score = best_match(row["LM_NAME"], candidates)

    if match is None or score < name_threshold:
        r = requests.get("https://en.wikipedia.org/w/api.php", params={
            "action": "query", "list": "search", "srsearch": row["LM_NAME"],
            "srlimit": 3, "format": "json"
        }, headers=HEADERS)
        text_candidates = [{"title": s["title"]} for s in r.json().get("query", {}).get("search", [])]
        match, score = best_match(row["LM_NAME"], text_candidates)
        confidence = "low" if match else "none"
    else:
        confidence = "high" if score > 85 else "medium"

    if match is None:
        return {"wiki_title": None, "wiki_url": None, "confidence": "none", "pageviews": None, "score":score}

    resolved = resolve_title_and_url(match["title"])
    views = get_pageviews(resolved["title"])
    return {
        "wiki_title": resolved["title"],
        "wiki_url": resolved["url"],
        "confidence": confidence,
        "pageviews": views,
        "score": score
    }

In [12]:
landmarks_gdf['neighborhood'] = landmarks_gdf['the_geom'].apply(find_neighborhood)

In [13]:
for idx, row in landmarks_gdf.iterrows():
    try:
        result = match_and_score(row, radius_m=100, name_threshold=50)
        landmarks_gdf.at[idx, 'wiki_title'] = result['wiki_title']
        landmarks_gdf.at[idx, 'wiki_url'] = result['wiki_url']
        landmarks_gdf.at[idx, 'confidence'] = result['confidence']
        landmarks_gdf.at[idx, 'pageviews'] = result['pageviews']
        landmarks_gdf.at[idx, 'score'] = result['score']
    except Exception as e:
        print(f"Error processing row {idx}: {e}")
    time.sleep(0.5) 

St. Mary's Church
Tarlac National High School
Lithuanian Alliance of America
Abraham E. Lefcourt
Daniel E. Barbey
List of New York City Designated Landmarks in Manhattan from 14th to 59th Streets
601 West 29th Street
Eiffel Tower
601 West 29th Street
Whitney Museum of American Art
Daniel Stern (actor)
Frederick Douglass Memorial Park
Brooklyn Friends School
Crown Building (Manhattan)
List of New York Public Library branches
Modulightor Building
Ulrich Franzen
African Burial Ground National Monument
Minton's Playhouse
Scribner Building
List of New York City Designated Landmarks in Queens
Los Angeles Fire Department
Bronx Opera House
New York City Fire Department
Simmons Colored School
Julius (restaurant)
Samuel Gompers High School
Lesbian Herstory Archives
Benjamin Ralph Kimlau
List of New York City Designated Landmarks in Staten Island
New York Public Library Main Branch
Holyrood Episcopal Church
Flatiron Building
List of New York City Designated Landmarks in Brooklyn
Guardian Angels
M

In [29]:
landmarks_gdf.head()

,the_geom,OBJECTID,Borough,Block,Lot,DESIG_ADDRESS,BBL,LM_NAME,LP_NUMBER,SITE_DESC,...,BCT2020,NTA2020,Shape_Leng,Shape_Area,neighborhood,wiki_title,wiki_url,confidence,pageviews,score
0,"POLYGON ((988453.531 200119.904, 988455.387 20...",1545,MN,341,26,440 GRAND STREET,1003410026,Church of Saint Mary,LP-02694,"Borough of Manhattan, Tax Map Block 341, Lot 26",...,1001402.0,MN0302,536.004671,14664.757287,Lower East Side,St. Mary's Church,https://en.wikipedia.org/wiki/St._Mary%27s_Church,low,6691.0,75.675676
1,"POLYGON ((989857.44 189382.342, 989822.282 189...",1544,BK,174,1201,362 SCHERMERHORN STREET,3001741201,Public School 15 Annex,LP-02696,"Borough of Brooklyn, Tax Map Block 174, Lot 1201",...,3003900.0,BK0202,257.191046,4003.793030,Downtown Brooklyn-DUMBO-Boerum Hill,Tarlac National High School,https://en.wikipedia.org/wiki/Tarlac_National_...,low,9395.0,40.816327
2,"POLYGON ((985459.65 212598.412, 985450.021 212...",1543,MN,754,34,307 WEST 30 STREET,1007540034,Lithuanian Alliance Building,LP-02695,"Borough of Manhattan Tax Map Block 754, Lot 34",...,1010300.0,MN0401,264.601114,2641.090847,Chelsea-Hudson Yards,Lithuanian Alliance of America,https://en.wikipedia.org/wiki/Lithuanian_Allia...,medium,NaN,72.413793
3,"POLYGON ((985969.69 210960.08, 985865.325 2110...",1542,MN,801,1,275 7 AVENUE,1008010001,Lefcourt Clothing Center,LP-02691,"Borough of Manhattan Tax Map Block 801, Lot 1",...,1009100.0,MN0401,645.143775,24309.040662,Chelsea-Hudson Yards,Abraham E. Lefcourt,https://en.wikipedia.org/wiki/Abraham_E._Lefcourt,low,2218.0,51.162791
4,"POLYGON ((988789.426 213157.93, 988762.083 213...",1541,MN,840,31,15 WEST 38 STREET,1008400031,Barbey Building,LP-02687,"Borough of Manhattan Tax Map Block 840, Lot 31",...,1008400.0,MN0502,327.013286,6051.234747,Midtown-Times Square,Daniel E. Barbey,https://en.wikipedia.org/wiki/Daniel_E._Barbey,low,3692.0,58.064516


In [30]:
from pathlib import Path

# Explicitly define the target path using raw string formatting (r"...")
target_dir = Path(r"c:\Users\seanc\OneDrive\Documents\Projects\NYC\nyc_landmarks\Landmarks")

# Force the operating system to create/verify the folder stream
target_dir.mkdir(parents=True, exist_ok=True)

# Combine the folder with your filename
final_file_path = target_dir / "landmarks_with_neighborhoods.csv"

# Save your data
landmarks_gdf.to_csv(final_file_path, index=False)
print(f"Successfully saved to OneDrive: {final_file_path}")

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\seanc\\OneDrive\\Documents\\Projects\\NYC\\nyc_landmarks\\Landmarks\\landmarks_with_neighborhoods.csv'

In [32]:
landmarks_gdf.to_csv("landmarks_with_neighborhoods.csv", index=False)

FileNotFoundError: [Errno 2] No such file or directory: 'landmarks_with_neighborhoods.csv'

### Checkpoint

Load the csv back into, start here to avoid running the wiki API again.

In [28]:
import os; print(os.getcwd())

c:\Users\seanc\OneDrive\Documents\Projects\NYC\nyc_landmarks\Landmarks


In [ ]:
# landmarks_gdf = pd.read_csv("output_csvs/landmarks_with_neighborhoods.csv")